# Solving the Travelling Salesman Problem using Genetic Algorithm

The *Traveling Salesman Problem* (TSP) is a classic and extremely important optimisation problem in the field of operations research and computer science. It can be stated as follows: *Given a set of cities and the distances between each pair of cities, the goal is to find the shortest possible route that visits each city exactly once and returns back to the starting city.* The TSP is known to be a NP-hard problem, hence it doesn't have a known polynomial-time algorithm which solves it exactly. In this first lab session, we will try to leverage *Genetic Algorithm* (GA) to come up with a solution.

Let's start by importing some useful modules.

In [3]:
import random
import numpy as np

We define a `City` class to handle our cities more easily. 

In [4]:
class City:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __repr__(self):
        return "(" + str(self.x) + ", " + str(self.y) + ")"
    
    def __eq__(self, other_city):
        return isinstance(other_city, City) and self.x == other_city.x and self.y == other_city.y
    
    def __ne__(self, other_city):
        return not self == other_city
    
    def distance(self, city):
        x_diff = abs(self.x - city.x)
        y_diff = abs(self.y - city.y)
        return np.sqrt((x_diff ** 2) + (y_diff ** 2))

Now we want a function to initialize our population. Remember that each route must contain each city **exactly once**.

In [5]:
def get_cities(n_cities, x_range, y_range):
    cities = []
    for _ in range(0, n_cities):
        city = City(x=int(random.random() * x_range), y=int(random.random() * y_range))
        if city.x != city.y:
            cities.append(city)
    return cities

def init_population(cities, pop_size):
    return [random.sample(cities, len(cities)) for _ in range(pop_size)]

In [ ]:
cities = get_cities(5, 5, 5)
init_pop = init_population(cities, 5)
print(cities)
for route in init_pop:
    print(route)


We define the **fitness** as the inverse of the route distance. Since we want to minimise the distance, we want to maximise our fitness score.

In [ ]:
def fitness_score(route):
    # fitness = 1 / total length of the closed tour
    d = sum(route[i].distance(route[(i+1) % len(route)]) for i in range(len(route)))
    return 1.0 / d if d > 0 else float('inf')

We define now a function to perform **tournament selection** (or any other selection strategy you may prefer).

In [ ]:
def tournament_selection(pop, fit, k):
    candidates = random.sample(pop, min(k, len(pop)))
    return max(candidates, key=fit)

Since each individual is a permutation of a set of different cities we must ensure that crossover and mutation generate valid individuals. As for the crossover, we can choose between the **partially mapped crossover** (**PMX**) and the **cycle crossover**. Let's implement both.

In [9]:
def pmx(parent1, parent2):
    n = len(parent1)
    a, b = sorted(random.sample(range(n + 1), 2))
    child = [None] * n
    child[a:b] = parent1[a:b]
    for i in range(a, b):
        gene = parent2[i]
        if gene in child[a:b]:
            continue
        pos = i
        while a <= pos < b:
            pos = parent2.index(parent1[pos])
        child[pos] = gene
    for i in range(n):
        if child[i] is None:
            child[i] = parent2[i]
    return child

In [ ]:
def cx(parent1, parent2):
    n = len(parent1)
    child = [None] * n
    i = 0
    while child[i] is None: # follow first cycle
        child[i] = parent1[i]
        i = parent1.index(parent2[i]) #remaining pos come from parent 2
    for j in range(n):
        if child[j] is None:
            child[j] = parent2[j]
    return child

Let's check if we did everything right with simple lists of integers

In [ ]:
parent1 = [7, 2, 3, 1, 5, 4, 6]
parent2 = [2, 4, 5, 6, 1, 7, 3]


print(pmx(parent1, parent2))
print(cx(parent1, parent2))

Also the mutation needs to generate valid individuals. Choose a strategy which suits our problem.

In [ ]:
def mutate(individual, p_m):
    ind = individual[:]
    if random.random() < p_m:
       i, j = sorted(random.sample(range(len(ind)), 2))
       ind[i:j+1] = reversed(ind[i:j+1])
    return ind

Now we define the function `generation`, which performs:
- Selection
- Crossover
- Mutation

One should have the possibility to include elitism.

In [ ]:
def generation(pop, fit, crossover, p_m, t_size, elitism=True, k_el=1):
  new_pop = sorted(pop, key=fit, reverse=True)[:k_el] if elitism else []
  while len(new_pop) < len(pop):
      p1 = tournament_selection(pop, fit, t_size)
      p2 = tournament_selection(pop, fit, t_size)
      new_pop.append(mutate(crossover(p1, p2), p_m))
  return new_pop

A GA performs a generational cycle a predefined number of times.

In [14]:
def get_best(pop, fit):
    best = max(pop, key=fit)
    return fit(best), best

In [ ]:
def GA(cities, pop_size, n, fit, crossover, t_size = 10, n_gen = 200,
       p_m=0.1, elitism=True, k_el=1):
    pop = init_population(cities, pop_size)
    history = []
    for _ in range(n_gen):
      pop = generation(pop, fit, crossover, p_m, t_size, elitism, k_el)
      history.append(get_best(pop, fit)[0])
    return get_best(pop, fit), history

We also implement an interactive version to visualize the evolution of the solution during the optimization process.

In [16]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

def plot_route(route, generation=None, fitness=None):
    # Extract x and y coordinates from the list of City objects
    x = [city.x for city in route]
    y = [city.y for city in route]
    
    # Append the first city to the end to close the TSP loop
    x.append(route[0].x)
    y.append(route[0].y)

    plt.figure(figsize=(8, 5))

    plt.scatter(x, y, color='b')

    plt.plot(x, y, linestyle='-', color='r', alpha=0.7)
    
    # Highlight the starting city
    plt.plot(x[0], y[0], marker='s', color='g', markersize=8, label='Start/End')

    # Dynamic title based on provided stats
    title = 'TSP Route'
    if generation is not None:
        title += f' | Gen: {generation}'
    if fitness is not None:
        title += f' | Fitness: {fitness:.6f}'

    plt.title(title)
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.legend()
    plt.show()

In [ ]:
def GA_animated(cities, pop_size, n, fit, crossover, t_size=10, n_gen=200, p_m=0.1, elitism=True, k_el=1):
    # CODE HERE
    # Use this snippet of code here in the main loop
    # if gen % 10 == 0 or gen == n_gen - 1:
    #   clear_output(wait=True)
    #   plot_route(best_route, generation=gen, fitness=best_fitness)
    pop = init_population(cities, pop_size)
    history = []
    for gen in range(n_gen):
        pop = generation(pop, fit, crossover, p_m, t_size, elitism, k_el)
        best_fitness, best_route = get_best(pop, fit)
        history.append(best_fitness)
        if gen % 10 == 0 or gen == n_gen - 1:
            clear_output(wait=True)
            plot_route(best_route, generation=gen, fitness=best_fitness)
    return (best_fitness, best_route), history

Run the Algorithm with different crossover startegies and parametrisations. For each experiment, make a line plot showing the best fitness score for each generation.

In [ ]:
cities = get_cities(25, 200, 200)

In [ ]:
# Run the animated version
res_animated, history_animated = GA_animated(
    cities=cities, 
    pop_size=41, 
    n=len(cities), 
    fit=fitness_score, 
    crossover=pmx,
    n_gen=200
)

In [ ]:
res, history = GA(cities=cities, pop_size=40, n=len(cities), fit=fitness_score, crossover=pmx)
best_fit, best_route = res

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(history)
plt.ylabel('Fitness')
plt.xlabel('Generation')
plt.show()

Let's plot the solution found by the GA. 

In [ ]:
route = res[1]

x = [city.x for city in route]
y = [city.y for city in route]

plt.scatter(x, y, label='Cities', color='blue')

for i in range(len(cities) - 1):
    plt.plot([x[i], x[i + 1]], [y[i], y[i + 1]], 'r-')

plt.plot([x[len(cities)-1], x[0]], [y[len(cities)-1], y[0]], 'r-')
plt.title('Solution')

# Show legend
plt.legend()

# Show the plot
plt.show()